# Every metric from one set of scores

The lesson's screening test flagged 130 of 1,000 patients, and the four counts behind that gave
every metric. Here the same kind of screen comes with the risk score behind each decision, so you
can build the confusion matrix at any threshold, compute precision, recall and F1, sweep the
threshold, and draw the ROC and precision-recall curves with the AUC. By the end you will see why
the everyone-healthy test's 95% accuracy means nothing.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

A screening model gave each of 1,000 patients a risk score between 0 and 1, and a follow-up exam
later settled who was actually sick: 50 patients, one in twenty. `y` is 1 for the sick and 0 for
the healthy, and `scores` holds the model's number for each. The screen flags
everyone at or above 0.5. Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)
n_sick, n_healthy = 50, 950
scores = np.concatenate([rng.beta(4, 2, n_sick), rng.beta(2, 5, n_healthy)])
y = np.concatenate([np.ones(n_sick, dtype=int), np.zeros(n_healthy, dtype=int)])
order = rng.permutation(len(y))                                       # shuffle the patients
scores, y = scores[order], y[order]

plt.hist(scores[y == 0], bins=30, range=(0, 1), alpha=0.6, label="healthy")
plt.hist(scores[y == 1], bins=30, range=(0, 1), alpha=0.6, label="sick")
plt.axvline(0.5, color="k", linestyle="--")
plt.xlabel("risk score")
plt.ylabel("patients")
plt.legend()
plt.show()
print(f"{len(y)} patients, {y.sum()} sick, {(scores >= 0.5).sum()} flagged at 0.5")

## Part 1 (coding task): The four counts

Flag every patient whose score is at or above the threshold. Each patient then lands in one of
four cells. A true positive is flagged and sick. A false positive is flagged and healthy. A false
negative is sick and not flagged. A true negative is healthy and not flagged. The four counts are
the confusion matrix, and every metric below is built from them.

Write `confusion_counts(y, scores, threshold)`. It returns `(TP, FP, FN, TN)` as Python integers.

In [ ]:
def confusion_counts(y, scores, threshold):
    """(TP, FP, FN, TN) when every score at or above the threshold is flagged."""
    # TODO
    raise NotImplementedError

In [ ]:
tiny_y = np.array([1, 0, 1, 0, 1, 0])
tiny_scores = np.array([0.9, 0.8, 0.4, 0.2, 0.5, 0.5])
counts = confusion_counts(tiny_y, tiny_scores, 0.5)
assert all(type(c) is int for c in counts), "return plain Python integers"
assert counts == (2, 2, 1, 1), f"at 0.5 two sick and two healthy are flagged, since 0.5 itself counts as flagged, got {counts}"
assert confusion_counts(tiny_y, tiny_scores, 0.0) == (3, 3, 0, 0), "a threshold of 0 flags everyone"
assert confusion_counts(tiny_y, tiny_scores, 1.1) == (0, 0, 3, 3), "a threshold above every score flags no one"
TP, FP, FN, TN = confusion_counts(y, scores, 0.5)
assert TP + FP + FN + TN == 1000 and TP + FN == 50, "the counts should add up to 1,000 patients and 50 sick"
assert (TP, FP) == (40, 91), f"at 0.5 the screen flags 40 sick patients and 91 healthy ones, got {TP} and {FP}"
print(f"at 0.5: TP {TP}  FP {FP}  FN {FN}  TN {TN}")
print("Part 1 passed")

## Part 2 (coding task): Precision, recall and F1

Precision is the fraction of flags that were right, $TP / (TP + FP)$. Recall is the fraction of
the sick who were caught, $TP / (TP + FN)$. F1 is their harmonic mean, $2PR / (P + R)$, which is
pulled toward the smaller of the two, so neither can be bought by abandoning the other. A test that
flags no one has no flags to be right about: treat any fraction with a zero denominator as 0
rather than crashing.

Write `precision_recall_f1(TP, FP, FN)`. It returns the three numbers as floats. The check includes
the lesson's screen, 40 flagged sick, 90 false alarms and 10 missed, which should give 0.308, 0.80
and 0.444.

In [ ]:
def precision_recall_f1(TP, FP, FN):
    """(precision, recall, f1), with 0 wherever a denominator is 0."""
    # TODO
    raise NotImplementedError

In [ ]:
p, r, f = precision_recall_f1(40, 90, 10)
assert abs(p - 0.3077) < 1e-3 and abs(r - 0.8) < 1e-9, f"the lesson's screen has precision 0.308 and recall 0.80, got {p:.3f} and {r:.3f}"
assert abs(f - 0.4444) < 1e-3, f"its F1 is 0.444, got {f:.3f}"
p, r, f = precision_recall_f1(50, 950, 0)
assert abs(p - 0.05) < 1e-9 and r == 1.0 and abs(f - 0.0952) < 1e-3, "flagging everyone: precision 0.05, recall 1, F1 0.095"
assert precision_recall_f1(0, 0, 50) == (0.0, 0.0, 0.0), "flagging no one should give three zeros, not a ZeroDivisionError"
p, r, f = precision_recall_f1(TP, FP, FN)
print(f"this screen at 0.5:  precision {p:.3f}  recall {r:.3f}  F1 {f:.3f}  accuracy {(TP + TN) / 1000:.3f}")
print(f"everyone healthy:    precision 0.000  recall 0.000  F1 0.000  accuracy {950 / 1000:.3f}")
print("Part 2 passed")

## Part 3 (coding task): Sweep the threshold

Nothing says the threshold has to be 0.5. Lower it and more patients are flagged: recall can only
rise and precision usually falls. Raise it and the reverse. Each threshold gives one confusion
matrix and one set of metrics, so sweep a grid of thresholds and watch all three move.

Write `sweep(y, scores, thresholds)`. It returns `(precision, recall, f1)`, three arrays with one
entry per threshold. The check plots them together with accuracy and reports the threshold with
the best F1.

In [ ]:
def sweep(y, scores, thresholds):
    """(precision, recall, f1) arrays, one entry per threshold."""
    # TODO
    raise NotImplementedError

In [ ]:
thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
P, R, F = sweep(y, scores, thresholds)
assert P.shape == R.shape == F.shape == (91,), "one entry per threshold"
assert np.all(np.diff(R) <= 0), "recall can only fall as the threshold rises"
assert np.allclose((P[45], R[45], F[45]), precision_recall_f1(TP, FP, FN)), "threshold 0.50 should match Part 2"
best_t = thresholds[int(np.argmax(F))]
assert abs(best_t - 0.67) < 1e-9, f"the best F1 should come at threshold 0.67, got {best_t:.2f}"

accuracy = np.array([(c[0] + c[3]) / 1000 for c in (confusion_counts(y, scores, t) for t in thresholds)])
plt.plot(thresholds, P, label="precision")
plt.plot(thresholds, R, label="recall")
plt.plot(thresholds, F, label="F1")
plt.plot(thresholds, accuracy, label="accuracy")
plt.axvline(best_t, color="k", linestyle="--")
plt.xlabel("threshold")
plt.legend()
plt.show()
print(f"best F1 {F.max():.3f} at threshold {best_t:.2f}, against {F[45]:.3f} at 0.50")
print("Part 3 passed")

## Part 4 (coding task): The ROC curve

The ROC curve shows every threshold at once. For each one, plot the true positive rate, which is
recall, against the false positive rate, $FP / (FP + TN)$. A threshold above every score flags no
one, the point $(0, 0)$. A threshold at the lowest score flags everyone, the point $(1, 1)$. A
model that ranks at random traces the diagonal, and a good one bows up toward the corner $(0, 1)$.

Write `roc_curve(y, scores)`. It returns `(fpr, tpr)`, two arrays that start at $(0, 0)$ and then
visit every distinct score as a threshold, from the highest score down. Sorting the distinct scores
and calling `confusion_counts` at each is enough.

In [ ]:
def roc_curve(y, scores):
    """(fpr, tpr) from no one flagged to everyone flagged, one point per distinct score."""
    # TODO
    raise NotImplementedError

In [ ]:
f_tiny, t_tiny = roc_curve(np.array([1, 0, 1, 0]), np.array([0.9, 0.8, 0.4, 0.2]))
assert np.allclose(f_tiny, [0, 0, 0.5, 0.5, 1]) and np.allclose(t_tiny, [0, 0.5, 0.5, 1, 1]), f"four patients give the points (0,0), (0,0.5), (0.5,0.5), (0.5,1), (1,1), got {list(zip(f_tiny, t_tiny))}"
fpr, tpr = roc_curve(y, scores)
assert fpr[0] == 0 and tpr[0] == 0, "the curve starts where no one is flagged"
assert fpr[-1] == 1 and tpr[-1] == 1, "the curve ends where everyone is flagged"
assert np.all(np.diff(fpr) >= 0) and np.all(np.diff(tpr) >= 0), "both rates can only rise as the threshold falls"
assert len(fpr) == len(np.unique(scores)) + 1, "one point per distinct score, plus the start"
fpr_half, tpr_half = FP / (FP + TN), TP / (TP + FN)
assert np.any((np.abs(fpr - fpr_half) < 1e-12) & (np.abs(tpr - tpr_half) < 1e-12)), "the 0.5 screen should be one point on the curve"

plt.plot(fpr, tpr, label="screening model")
plt.plot([0, 1], [0, 1], "k--", label="random ranking")
plt.scatter([fpr_half], [tpr_half], color="r", zorder=3, label="threshold 0.5")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate (recall)")
plt.legend()
plt.show()
print("Part 4 passed")

## Part 5 (coding task): The area under the curve

The AUC is the area under the ROC curve: 1 for a perfect ranker and 0.5 for a random one. It is
also the probability that a randomly chosen sick patient scores higher than a randomly chosen
healthy one, a statement about ranking that involves no threshold at all. The two definitions
agree exactly when no two patients share a score, and the check uses the second
to test the first.

Write `auc(fpr, tpr)`. It returns the area under the curve by the trapezoid rule: for each pair of
consecutive points, the width in `fpr` times the average of the two `tpr` values, summed.

In [ ]:
def auc(fpr, tpr):
    """Area under the curve by the trapezoid rule, as a float."""
    # TODO
    raise NotImplementedError

In [ ]:
assert abs(auc(np.array([0.0, 1.0]), np.array([0.0, 1.0])) - 0.5) < 1e-12, "the diagonal has area 0.5"
assert abs(auc(np.array([0.0, 0.0, 1.0]), np.array([0.0, 1.0, 1.0])) - 1.0) < 1e-12, "straight up then across has area 1"
assert abs(auc(f_tiny, t_tiny) - 0.75) < 1e-12, "the four-patient curve has area 0.75"
area = auc(fpr, tpr)
sick_scores, healthy_scores = scores[y == 1], scores[y == 0]
pairs = np.mean(sick_scores[:, None] > healthy_scores[None, :])
assert abs(area - pairs) < 1e-9, f"the area {area:.4f} should equal the fraction of sick-healthy pairs ranked correctly, {pairs:.4f}"
print(f"AUC {area:.3f}: a random sick patient outscores a random healthy one {100 * pairs:.1f}% of the time")
print("Part 5 passed")

## Part 6 (coding task): The precision-recall curve

With 950 healthy patients, the false positive rate barely moves while dozens of false alarms
pile up, so the ROC curve flatters the screen. The precision-recall curve divides by the number of
flags instead and is the honest picture when positives are rare. Plot precision against recall for
every threshold, from the single highest score down. A random ranker sits near the positive rate,
0.05, not 0.5.

Write `pr_curve(y, scores)`. It returns `(recall, precision)`, one point per distinct score, from
the highest score down, so recall runs from near 0 up to 1.

In [ ]:
def pr_curve(y, scores):
    """(recall, precision) at every distinct score as the threshold, highest threshold first."""
    # TODO
    raise NotImplementedError

In [ ]:
r_tiny, p_tiny = pr_curve(np.array([1, 0, 1, 0]), np.array([0.9, 0.8, 0.4, 0.2]))
assert np.allclose(r_tiny, [0.5, 0.5, 1.0, 1.0]) and np.allclose(p_tiny, [1.0, 0.5, 2 / 3, 0.5]), f"four patients give recall (0.5, 0.5, 1, 1) and precision (1, 0.5, 0.667, 0.5), got {r_tiny} and {p_tiny}"
rec, prec = pr_curve(y, scores)
assert len(rec) == len(np.unique(scores)), "one point per distinct score"
assert np.all(np.diff(rec) >= 0), "recall should rise as the threshold falls"
assert rec[-1] == 1.0 and abs(prec[-1] - 0.05) < 1e-12, "flagging everyone gives recall 1 and precision 0.05, the positive rate"
assert prec[0] == 1.0 and abs(rec[0] - 0.02) < 1e-12, "the single highest score belongs to a sick patient, so the first point is recall 1/50 at precision 1"

plt.plot(rec, prec, label="screening model")
plt.axhline(0.05, color="k", linestyle="--", label="random ranking")
plt.xlabel("recall")
plt.ylabel("precision")
plt.ylim(0, 1.05)
plt.legend()
plt.show()
print("Part 6 passed")

## Solution

### Part 1: The four counts

In [ ]:
def confusion_counts(y, scores, threshold):
    """(TP, FP, FN, TN) when every score at or above the threshold is flagged."""
    flagged = scores >= threshold
    sick = y == 1
    TP = int(np.sum(flagged & sick))
    FP = int(np.sum(flagged & ~sick))
    FN = int(np.sum(~flagged & sick))
    TN = int(np.sum(~flagged & ~sick))
    return TP, FP, FN, TN

### Part 2: Precision, recall and F1

In [ ]:
def precision_recall_f1(TP, FP, FN):
    """(precision, recall, f1), with 0 wherever a denominator is 0."""
    precision = TP / (TP + FP) if TP + FP > 0 else 0.0
    recall = TP / (TP + FN) if TP + FN > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return float(precision), float(recall), float(f1)

### Part 3: Sweep the threshold

In [ ]:
def sweep(y, scores, thresholds):
    """(precision, recall, f1) arrays, one entry per threshold."""
    rows = [precision_recall_f1(*confusion_counts(y, scores, t)[:3]) for t in thresholds]
    precision, recall, f1 = np.array(rows).T
    return precision, recall, f1

### Part 4: The ROC curve

In [ ]:
def roc_curve(y, scores):
    """(fpr, tpr) from no one flagged to everyone flagged, one point per distinct score."""
    fpr, tpr = [0.0], [0.0]
    for t in np.unique(scores)[::-1]:
        TP, FP, FN, TN = confusion_counts(y, scores, t)
        fpr.append(FP / (FP + TN))
        tpr.append(TP / (TP + FN))
    return np.array(fpr), np.array(tpr)

### Part 5: The area under the curve

In [ ]:
def auc(fpr, tpr):
    """Area under the curve by the trapezoid rule, as a float."""
    widths = np.diff(fpr)
    heights = (tpr[1:] + tpr[:-1]) / 2
    return float(np.sum(widths * heights))

### Part 6: The precision-recall curve

In [ ]:
def pr_curve(y, scores):
    """(recall, precision) at every distinct score as the threshold, highest threshold first."""
    recall, precision = [], []
    for t in np.unique(scores)[::-1]:
        TP, FP, FN, TN = confusion_counts(y, scores, t)
        p, r, _ = precision_recall_f1(TP, FP, FN)
        recall.append(r)
        precision.append(p)
    return np.array(recall), np.array(precision)